# Running Segger
Runs Segger on all samples and post-processes output into AnnData + SpatialData.

**Inputs:** `transcripts.parquet` + `nucleus_boundaries.parquet` per sample  
**Outputs:** `segger_spatialdata.zarr` per sample (under `{no|with}_SC_reference/`)  
**Env:** `segger-incremental` conda environment (required)  
**Note:** run the notebook twice — once with `single_cell_used="no"`, once with `"with"` — to produce both outputs

`single_cell_used` controls both the Segger mode and the output subfolder name (`no_SC_reference/` or `with_SC_reference/`). Set it before running.

In [ ]:
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
single_cell_used = "no"  # "no" = unguided by scRNA | "with" = scRNA-guided; also names the output subfolder
INPUT_DIR = Path("")  # folder containing per-sample subdirs with boundaries and transcripts
SEGGER_OUTPUT_BASE_DIR = Path("")  # base output dir; subdir {single_cell_used}_SC_reference is created automatically
SCRNA_REFERENCE_PATH = Path("")  # path to scRNA .h5ad reference (only needed if single_cell_used == "with")

# derived path (do not edit)
base_output_dir = SEGGER_OUTPUT_BASE_DIR / f"{single_cell_used}_SC_reference"


In [ ]:
import os
import subprocess

base_output_dir.mkdir(parents=True, exist_ok=True)

my_env = os.environ.copy()
my_env["CUDA_VISIBLE_DEVICES"] = "1"  # adjust to the GPU index you want to use

for folder_name in os.listdir(INPUT_DIR):
    input_folder_path = INPUT_DIR / folder_name

    if input_folder_path.is_dir():
        sample_id = folder_name
        sample_output_dir = base_output_dir / sample_id
        sample_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Starting Segger for: {sample_id}")

        # Include --scrna-reference-path only in scRNA-guided mode
        if single_cell_used == "with":
            command = [
                "segger", "segment",
                "--min-qv", "0",
                "--node-representation-dim", "60",
                "--prediction-mode", "nucleus",
                "--scrna-reference-path", str(SCRNA_REFERENCE_PATH),
                "-i", str(input_folder_path),
                "-o", str(sample_output_dir)
            ]
        else:
            command = [
                "segger", "segment",
                "--min-qv", "0",
                "--node-representation-dim", "60",
                "--prediction-mode", "nucleus",
                "-i", str(input_folder_path),
                "-o", str(sample_output_dir)
            ]

        try:
            print(f"Running Segger on {sample_id}...")
            result = subprocess.run(
                command,
                env=my_env,
                check=True,
                capture_output=True,
                text=True
            )
            print("Success!")
        except subprocess.CalledProcessError as e:
            print(f"Error: {e.stderr}")


And now we need to get Anndata object, there is Seger export option, but it is not exactly what we need. Instead
we first join the initial transcritps table with the predicted one, bcs the predicted one has only cell_id no location data.
Joining is done according to row indees, which remanin preserved 

In [ ]:
def segger2adata(segger_output_folder, initial_transcripts_folder):
    # Joins Segger's cell-assignment parquet with the original transcripts parquet
    # (which carries spatial coordinates) to build a cell × gene AnnData.
    # Merge is on row_index, which Segger preserves from the input file.
    from shapely.geometry import MultiPoint
    import anndata as ad
    import pandas as pd

    predictions = pd.read_parquet(segger_output_folder / "segger_segmentation.parquet")
    input_tx = pd.read_parquet(initial_transcripts_folder / "transcripts.parquet")

    # predictions has cell_id per row; input_tx has coordinates — merge by preserved row index
    merged = pd.merge(predictions, input_tx, left_on="row_index", right_index=True, how="left")

    merged = merged[merged['segger_cell_id'].notna()]
    cell_counts = merged['segger_cell_id'].value_counts()
    valid_cells = cell_counts[cell_counts >= 2].index
    merged = merged[merged['segger_cell_id'].isin(valid_cells)]

    cell_gene_matrix = pd.crosstab(merged['segger_cell_id'], merged['gene'])

    # Centroids are the centroid of the convex hull of each cell's assigned transcript coordinates
    centroids = (
        merged
        .groupby('segger_cell_id')
        .apply(lambda g: MultiPoint(g[['global_x', 'global_y']].to_numpy()).convex_hull.centroid, include_groups=False)
    )
    centroids_df = centroids.apply(lambda p: (p.x, p.y)).to_list()
    centroids_df = pd.DataFrame(centroids_df, index=centroids.index, columns=['centroid_x', 'centroid_y'])

    cell_gene_matrix = centroids_df.join(cell_gene_matrix, on='segger_cell_id')

    obs = cell_gene_matrix[['centroid_x', 'centroid_y']]
    X = cell_gene_matrix.drop(columns=['centroid_x', 'centroid_y'])
    X = X.fillna(0)
    var = pd.DataFrame(index=X.columns)

    adata = ad.AnnData(X=X, obs=obs, var=var)

    # SegTraQ requires cell_id column and region as a category in obs
    adata.obs["cell_id"] = adata.obs.index.astype(int)
    adata.obs["region"] = "cell_boundaries"
    adata.obs["region"] = adata.obs["region"].astype("category")
    adata.obsm['spatial'] = obs[['centroid_x', 'centroid_y']].values

    merged['cell_id'] = merged['segger_cell_id']

    merged.to_parquet(segger_output_folder / "merged_segger_segmentation.parquet")
    adata.write_h5ad(segger_output_folder / "segger_adata.h5ad")
    print(f"finished converting {segger_output_folder} to anndata object")


Main script for conversion of segger parquet output file into Anndata object

In [ ]:
import os
from pathlib import Path

segger_output_folder = SEGGER_OUTPUT_BASE_DIR / f"{single_cell_used}_SC_reference"
segger_input_folder = INPUT_DIR

for folder_name in os.listdir(segger_output_folder):
    sample_folder_path = segger_output_folder / folder_name
    #print(sample_folder_path)
    #print(folder_name)

    segger2adata(sample_folder_path, segger_input_folder / folder_name)


Furher postprocessing for Segtraq compatibility

In [ ]:
def cell_to_polygon(g):
    import geopandas as gpd
    from shapely.geometry import MultiPoint
    return MultiPoint(g[['global_x', 'global_y']].to_numpy()).convex_hull  #same method as in 2cellbygene.ipynb

In [ ]:
def to_spatialdata(sample, input_folder):
    # Wraps the Segger AnnData into a SpatialData .zarr store.
    # Since Segger produces no boundary file, cell shapes are built as convex hulls
    # of each cell's assigned transcript coordinates.
    import spatialdata as sd
    import anndata as ad
    import pandas as pd
    import geopandas as gpd

    adata = ad.read_h5ad(f'{input_folder}/{sample}/segger_adata.h5ad')
    transcripts_df = pd.read_parquet(f'{input_folder}/{sample}/merged_segger_segmentation.parquet')

    # Build cell boundary polygons from transcript point clouds
    cell_shapes = (
        transcripts_df
        .groupby("cell_id")
        .apply(cell_to_polygon)
    )
    cell_shapes_gdf = gpd.GeoDataFrame(geometry=cell_shapes, index=cell_shapes.index)
    cell_shapes_gdf.index.name = "cell_id"

    # Rename columns to SpatialData convention
    transcripts_df = transcripts_df.rename(columns={
        "gene": "feature_name",
        "global_x": "x",
        "global_y": "y",
        "global_z": "z"
    })
    transcripts_df["feature_name"] = transcripts_df["feature_name"].astype("category")

    # Align obs_names with cell_shapes_gdf index (required by SpatialData TableModel)
    adata.obs["cell_id"] = cell_shapes_gdf.index.astype(str)
    cell_shapes_gdf.index = cell_shapes_gdf.index.astype(str)
    adata.obs_names = adata.obs["cell_id"]

    print(cell_shapes_gdf.geometry.geom_type.value_counts())

    sdata = sd.SpatialData(
        points={"transcripts": sd.models.PointsModel.parse(transcripts_df)},
        shapes={"cell_boundaries": sd.models.ShapesModel.parse(cell_shapes_gdf)},
        tables={
            "table": sd.models.TableModel.parse(
                adata,
                region_key="region",
                region="cell_boundaries",
                instance_key="cell_id"
            )
        },
    )
    sdata.write(f"{input_folder}/{sample}/segger_spatialdata.zarr", overwrite=True)

In [20]:
single_cell_used = 'no' # 'no' / 'with'

In [ ]:
# --CONFIG--
segger_output_folder = SEGGER_OUTPUT_BASE_DIR / f"{single_cell_used}_SC_reference"

for sample in os.listdir(segger_output_folder):
    print(f"Processing sample: {sample}")
    to_spatialdata(sample, str(segger_output_folder))


This is the end of postprocessing SEgger ouputs, next step is integrating all data in Segtraq anaylsis